# Do You Need a Vector Database for AI Agent Memory? FAISS vs Amazon S3 Vectors

The traveler from [Demo 01](../01-key-value-memory-demo/) is back with a season of
memories. They ask:

> *"What should I avoid eating when I go out for dinner on this trip?"*

The answer is in memory, under the key `dietary_notes`, but the question names no
key and shares no words with the note ("eating" vs "vegetarian / shellfish"). That
is the line this notebook measures:

| You know... | Use |
|-------------|-----|
| the **key** ("what's my preferred cabin?") | key-value memory (Demo 01): exact, no embeddings |
| only the **meaning** | vector memory: embed the memories once, retrieve by similarity |

Once you need vector memory, the question devs actually search for: do you need a
vector *database*, or is an in-process index enough? Two backends, same
[Titan V2](https://docs.aws.amazon.com/bedrock/latest/userguide/titan-embedding-models.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el)
embeddings, same memories:

- **FAISS**: the index lives in RAM. Microsecond queries, no infrastructure, dies with the process.
- **Amazon S3 Vectors**: the index lives in a vector bucket. You create bucket + index (the notebook self-provisions both), it survives restarts and is reachable from any process.

`embed()`, the three stores, and the recall tools are defined in the cells below.
Only the AWS session helper and a timing utility are imported.

## Install dependencies

In [ ]:
%pip install -q -r requirements.txt

## Configure credentials

- AWS credentials (`aws configure`) power both Titan embeddings (Bedrock) and S3 Vectors. The vector bucket + index are created automatically if missing.
- `OPENAI_API_KEY` is only for the agent conversation at the end (the retrieval measurements need no LLM).

In [ ]:
import os

# Bearer-token env vars would override the AWS profile; drop them before boto3 loads.
os.environ.pop('AWS_BEARER_TOKEN', None)
os.environ.pop('AWS_BEARER_TOKEN_BEDROCK', None)

from dotenv import load_dotenv
load_dotenv()

assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env (needed for the agent at the end)'

## The embedder and the three stores

`embed()` is one Titan V2 call (1024 dims), used by both vector backends so the
only thing that varies is the store. The three stores follow one shape: a
key-value dict (exact lookup, blind to meaning), a FAISS index (cosine similarity
in RAM), and an S3 Vectors store (the same similarity, persisted in a vector
bucket). The AWS session helper is imported so a stray env token can't hijack the
profile; everything else is here in the cell.

In [ ]:
import json
import time

import boto3
import faiss
import numpy as np

# _aws builds a boto3 Session from AWS_PROFILE (or the default chain), so env
# bearer tokens can't override it. Imported because it is credential plumbing.
from memory_stores import _aws

EMBED_MODEL_ID = "amazon.titan-embed-text-v2:0"
EMBED_DIM = 1024
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
VECTOR_BUCKET = os.getenv("VECTOR_BUCKET", "agent-memory-demo-vectors")
VECTOR_INDEX = os.getenv("VECTOR_INDEX", "traveler-memories")


def embed(text: str) -> list[float]:
    """Real Titan V2 embedding (1024 dims), used by both vector backends."""
    client = _aws().client("bedrock-runtime", region_name=AWS_REGION)
    resp = client.invoke_model(
        modelId=EMBED_MODEL_ID,
        body=json.dumps({"inputText": text, "dimensions": EMBED_DIM}),
    )
    return json.loads(resp["body"].read())["embedding"]


def timed(fn, *args, **kwargs):
    """Run fn and return (result, elapsed_ms). The demo measures, never guesses."""
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    return result, (time.perf_counter() - start) * 1000


class KeyValueStore:
    """Plain dict: exact when you know the key, blind to meaning."""

    def __init__(self):
        self.data = {}

    def put(self, key, text):
        self.data[key] = text

    def get(self, key):
        return self.data.get(key)

    def keyword_search(self, query):
        """The best a key-value store can do without a key: substring matching."""
        words = {w.lower().strip("?.,!") for w in query.split() if len(w) > 3}
        return [text for text in self.data.values()
                if any(w in text.lower() for w in words)]

    def dump_all(self):
        return "\n".join(self.data.values())


class FaissStore:
    """Vector memory in RAM: cosine similarity via a normalized inner-product index."""

    def __init__(self):
        self.index = faiss.IndexFlatIP(EMBED_DIM)
        self.texts = []

    def put(self, text, vector):
        v = np.array([vector], dtype="float32")
        faiss.normalize_L2(v)
        self.index.add(v)
        self.texts.append(text)

    def query(self, vector, top_k=3):
        v = np.array([vector], dtype="float32")
        faiss.normalize_L2(v)
        scores, ids = self.index.search(v, top_k)
        return [(self.texts[i], float(s)) for i, s in zip(ids[0], scores[0]) if i >= 0]


class S3VectorStore:
    """Vector memory in a vector bucket: persists across restarts, shared across processes."""

    def __init__(self, bucket=VECTOR_BUCKET, index=VECTOR_INDEX):
        self.bucket = bucket
        self.index = index
        self.client = _aws().client("s3vectors", region_name=AWS_REGION)
        self._ensure()

    def _ensure(self):
        """Create the vector bucket and index if they don't exist (idempotent)."""
        try:
            self.client.get_vector_bucket(vectorBucketName=self.bucket)
        except self.client.exceptions.NotFoundException:
            self.client.create_vector_bucket(vectorBucketName=self.bucket)
            print(f"  created vector bucket {self.bucket}")
        try:
            self.client.get_index(vectorBucketName=self.bucket, indexName=self.index)
        except self.client.exceptions.NotFoundException:
            self.client.create_index(
                vectorBucketName=self.bucket, indexName=self.index,
                dimension=EMBED_DIM, distanceMetric="cosine", dataType="float32",
            )
            print(f"  created vector index {self.index} ({EMBED_DIM} dims, cosine)")

    def put(self, key, text, vector):
        self.client.put_vectors(
            vectorBucketName=self.bucket, indexName=self.index,
            vectors=[{"key": key, "data": {"float32": vector}, "metadata": {"text": text}}],
        )

    def query(self, vector, top_k=3):
        resp = self.client.query_vectors(
            vectorBucketName=self.bucket, indexName=self.index,
            queryVector={"float32": vector}, topK=top_k,
            returnDistance=True, returnMetadata=True,
        )
        return [(v["metadata"]["text"], 1.0 - v["distance"]) for v in resp.get("vectors", [])]

    def count(self):
        resp = self.client.list_vectors(vectorBucketName=self.bucket, indexName=self.index)
        return len(resp.get("vectors", []))

    def clear(self):
        """Delete this demo's vectors so reruns start empty (bucket/index stay)."""
        resp = self.client.list_vectors(vectorBucketName=self.bucket, indexName=self.index)
        keys = [v["key"] for v in resp.get("vectors", [])]
        if keys:
            self.client.delete_vectors(vectorBucketName=self.bucket, indexName=self.index, keys=keys)
        return len(keys)


print('embedder + three stores defined')

## The traveler's memories

Ten notes from past conversations. `dietary_notes` shares no words with the
question we will ask, which is what makes the keyword scan fail and vector search
succeed. Each note is written to all three stores, embedded once.

In [ ]:
TRAVELER_NOTES = {
    'dietary_notes': 'Vegetarian; severe shellfish allergy, strictly no crustaceans or mollusks.',
    'preferred_cabin': 'Books business class on flights longer than six hours.',
    'airline_status': 'Oneworld Emerald via Iberia Plus; prefers Iberia when fares are close.',
    'layover_rule': 'Refuses overnight connections; anything over four hours is too long.',
    'seat_choice': 'Aisle seat, as far forward as possible, never next to the lavatory.',
    'travel_season': 'Tries to fly shoulder season, late September or early May.',
    'hotel_loyalty': 'No hotel program; picks small independent places near the old town.',
    'packing_habit': 'Carry-on only, no matter the trip length.',
    'budget_ceiling': 'Keeps round-trip fares under 1,500 USD unless it is a special occasion.',
    'airport_home': 'Based near JFK; can use EWR if the fare difference is over 200 USD.',
}

QUESTION = 'What should I avoid eating when I go out for dinner on this trip?'

kv = KeyValueStore()
faiss_store = FaissStore()
s3v = S3VectorStore()   # self-provisions the vector bucket + index if missing
s3v.clear()             # rerun-safe: drop this demo's old vectors

for key, note in TRAVELER_NOTES.items():
    kv.put(key, note)
    vector = embed(note)
    faiss_store.put(note, vector)
    s3v.put(key, note, vector)

print(f'stored {len(TRAVELER_NOTES)} memories in all three stores')

---
## Test 1. The key-value limit

Without a key, a key-value store can only scan for shared words or dump everything
into context. The scan misses (no shared words); dump-all pays for the whole memory
on every question.

In [ ]:
hits = kv.keyword_search(QUESTION)
found = any('shellfish' in h for h in hits)
print(f'keyword scan hits: {len(hits)} (answer found: {found})')
for h in hits:
    print('  -', h[:70])

dump = kv.dump_all()
print(f'\nfallback dump-all: {len(dump):,} chars of memory into context, every question')

---
## Test 2. FAISS: retrieval by meaning, in-process

Same memories, embedded once with Titan V2. The question embeds to a vector;
cosine similarity finds `dietary_notes` even though no meaningful word overlaps.
FAISS answers in microseconds — it is a RAM index.

In [ ]:
qvec, embed_ms = timed(embed, QUESTION)
faiss_store.query(qvec, 3)   # warm-up: first call pays one-time setup
hits, query_ms = timed(faiss_store.query, qvec, 3)

for text, score in hits:
    print(f'  {score:.3f}  {text[:70]}')
print(f'\nembed: {embed_ms:.0f} ms | FAISS query: {query_ms:.2f} ms')

---
## Test 3. Amazon S3 Vectors: the index that survives

Same memories, same embeddings, but the index lives in a vector bucket, not in
RAM. The query is a network call (`query_vectors`), so it costs milliseconds
instead of microseconds. What you buy is persistence: a new client object
(nothing in RAM, the "restart") still sees every vector.

In [ ]:
s3v.query(qvec, 3)           # warm-up: first call pays TLS/connection setup
hits, query_ms = timed(s3v.query, qvec, 3)

for text, score in hits:
    print(f'  {score:.3f}  {text[:70]}')
print(f'\nS3 Vectors query: {query_ms:.0f} ms')

fresh = S3VectorStore()      # new client, nothing in RAM: the 'restart'
print(f"fresh client sees {fresh.count()}/{len(TRAVELER_NOTES)} vectors: index survived")

---
## Test 4. A real agent choosing between the tools

The agent gets both recall tools. Their docstrings say when each applies (key
lookup for known identifiers, semantic for meaning), so the agent picks per
question. Both tools are defined here; `recall_semantic` queries whichever vector
store you pass in, so the backend is interchangeable from the agent's view.

In [ ]:
from strands import tool


@tool
def recall_by_key(key: str) -> str:
    """Look up one memory when you KNOW its exact key.

    Args:
        key: the exact key used at write time, e.g. "dietary_notes".
    """
    text = kv.get(key)
    return text or f"No memory stored under '{key}'."


@tool
def recall_semantic(question: str, top_k: int = 3) -> str:
    """Retrieve the memories most relevant to a question BY MEANING.

    The stored notes need not share any words with the question.

    Args:
        question: the user's question, as asked.
        top_k: how many memories to return (default 3).
    """
    hits = s3v.query(embed(question), top_k)
    if not hits:
        return "No relevant memories found."
    return json.dumps([{"note": t, "score": round(s, 3)} for t, s in hits], indent=1)


os.environ['OTEL_SDK_DISABLED'] = 'true'
from strands import Agent
from strands.models.openai import OpenAIModel

MODEL = OpenAIModel(model_id='gpt-4o-mini')

# Amazon Bedrock instead (uses your AWS credentials, no OpenAI key):
# from strands.models import BedrockModel
# MODEL = BedrockModel(model_id='openai.gpt-oss-120b-1:0', region_name='us-west-2')

agent = Agent(
    model=MODEL,
    system_prompt=('You are a travel assistant with long-term memory of this user. '
                   'Be concise: 2-3 sentences maximum.'),
    tools=[recall_by_key, recall_semantic],
    callback_handler=None,
)

resp = agent(QUESTION)
print(f'User: {QUESTION}\nAgent: {str(resp).strip()}')

resp = agent('And what is my preferred cabin?')
print(f'\nUser: And what is my preferred cabin?\nAgent: {str(resp).strip()}')

---
### Deterministic vs model-based here

The control lives in the agent's harness: the recall tools the agent calls.

Deterministic (plain code): the keyword scan, the cosine similarity search over the
vectors, and the stores themselves. Model-based: the **embedding** (Titan V2 maps text
to a vector) and the agent picking a tool. An embedding is a model call, not arithmetic,
neural-network inference on GPUs varies with floating-point non-associativity across runs
([Enabling Determinism in LLM Inference](https://arxiv.org/abs/2601.17768), 2026). The
cosine math *over* those vectors is deterministic; producing the vectors is not.

---
## The decision table

| You need | Pick | Why |
|----------|------|-----|
| Facts under known keys (profile, prefs) | **Key-value** (Demo 01) | Exact and instant. No embeddings for lookups |
| Search by meaning, single process, prototype | **FAISS** | Microsecond queries, no infrastructure. Gone when the process dies |
| Search by meaning, persistent, shared | **Amazon S3 Vectors** | Milliseconds per query, nothing to administer, survives restarts, any process can read it |
| Multi-hop questions over relationships | **Graph** → [Demo 03](../03-graph-memory-demo/) | Similarity can't follow edges |

Two footnotes from the measurements: the embedding call dominates (~0.5 s per
question with Titan V2, the same for both backends), and for 10 memories a
dump-all is still cheap. Vector memory earns its keep as memory grows to hundreds
of notes, where dump-all costs thousands of tokens per question.

Production note: for multi-tenant SaaS memory on S3 Vectors, see the
[`strands-s3-vectors-memory`](https://github.com/aws-samples/data-for-saas-patterns/tree/main/samples/multi-tenant-strands-s3-vectors-memory)
community plugin (one index per tenant with IAM-scoped credentials).

---
## Cleanup: delete AWS resources (optional)

Remove the S3 Vectors index and bucket this notebook created. The self-provisioning
code recreates them next run, so cleanup is optional. The bucket is deleted only if
no other index remains in it (other demos may share it).

In [ ]:
import boto3, os
from dotenv import load_dotenv
load_dotenv()

AWS_REGION = os.getenv('AWS_REGION', 'us-east-1')
profile = os.getenv('AWS_PROFILE')
session = boto3.Session(profile_name=profile) if profile else boto3.Session()

bucket = os.getenv('VECTOR_BUCKET', 'agent-memory-demo-vectors')
index  = os.getenv('VECTOR_INDEX', 'traveler-memories')
s3v = session.client('s3vectors', region_name=AWS_REGION)

# Delete this demo's vectors and index.
try:
    resp = s3v.list_vectors(vectorBucketName=bucket, indexName=index)
    keys = [v['key'] for v in resp.get('vectors', [])]
    if keys:
        s3v.delete_vectors(vectorBucketName=bucket, indexName=index, keys=keys)
        print(f"  deleted {len(keys)} vector(s) from index '{index}'")
    s3v.delete_index(vectorBucketName=bucket, indexName=index)
    print(f"  deleted index '{index}'")
except s3v.exceptions.NotFoundException:
    print(f"  index '{index}' not found (already deleted or never created)")

# Delete the bucket only if empty (other demos may share it).
try:
    remaining = s3v.list_indexes(vectorBucketName=bucket).get('indexes', [])
    if remaining:
        names = [i['indexName'] for i in remaining]
        print(f"  bucket '{bucket}' still has indexes {names}, leaving it")
    else:
        s3v.delete_vector_bucket(vectorBucketName=bucket)
        print(f"  deleted S3 Vectors bucket '{bucket}' (was empty)")
except s3v.exceptions.NotFoundException:
    print(f"  bucket '{bucket}' not found (already deleted or never created)")